# Fase 0 — Swing points

Primer concepto a formalizar: los swing highs/lows que alimentan BOS/CHOCH, order blocks y el resto de la estructura.

Usamos `smartmoneyconcepts.smc.swing_highs_lows(ohlc, swing_length)` como heurística inicial (alto recall, parámetro `swing_length` ajustable) en vez de reinventar la detección desde cero. El objetivo de este notebook es generar ejemplos reales anotados, revisarlos juntos, y ajustar `swing_length` hasta que coincida con tu criterio — el resultado final se registra en `config/strategy_rules.yaml` y `docs/rule_decisions.md`.

**Cómo iterar**: cambia `SYMBOL`, `TIMEFRAME` o `SWING_LENGTH` en la celda de configuración, vuelve a correr el notebook, y revisa las imágenes generadas en `notebooks/00_rule_spec/output/swing_points/`.

In [1]:
import os
import sys
from pathlib import Path

# Asegura que las rutas relativas de trading_ai.data.store apunten a la raíz del proyecto,
# sin importar desde dónde se haya lanzado Jupyter.
PROJECT_ROOT = Path.cwd().parents[1] if (Path.cwd().name == "00_rule_spec") else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
import mplfinance as mpf
import matplotlib.pyplot as plt
from smartmoneyconcepts import smc

from trading_ai.data import store

Thank you for using SmartMoneyConcepts! ⭐ Please show your support by giving a star on the GitHub repository: https://github.com/joshyattridge/smart-money-concepts


In [2]:
# --- Configuración: ajusta esto entre iteraciones ---
SYMBOL = "EURUSD"
TIMEFRAME = "1H"
SWING_LENGTH = 10  # candados a izquierda/derecha para confirmar un swing

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "00_rule_spec" / "output" / "swing_points"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_EXAMPLES = 25  # cuántos ejemplos renderizar para revisión
BARS_OF_CONTEXT = 60  # velas a cada lado del swing point en la imagen

In [3]:
from datetime import datetime, timezone

df = store.get_ohlcv(SYMBOL, TIMEFRAME, datetime(2000, 1, 1, tzinfo=timezone.utc), datetime.now(timezone.utc))
print(f"{SYMBOL}/{TIMEFRAME}: {len(df)} velas, de {df.index.min()} a {df.index.max()}")
df.tail(3)

EURUSD/1H: 18630 velas, de 2023-07-05 05:00:00+00:00 a 2026-07-03 23:00:00+00:00


,open,high,low,close,volume
time,,,,,
2026-07-03 21:00:00+00:00,1.14404,1.14415,1.14380,1.14389,6474
2026-07-03 22:00:00+00:00,1.14389,1.14407,1.14356,1.14363,3950
2026-07-03 23:00:00+00:00,1.14363,1.14377,1.14334,1.14357,1762


In [4]:
# Detección con la heurística inicial (smartmoneyconcepts), alto recall vía swing_length bajo.
# swing_highs_lows devuelve un DataFrame con índice posicional (0..n-1), no el datetime de df,
# así que lo realineamos al índice real de df para poder ubicar cada swing en el tiempo.
swings = smc.swing_highs_lows(df, swing_length=SWING_LENGTH)
swings.index = df.index
swing_points = swings.dropna(subset=["HighLow"])
print(f"{len(swing_points)} swing points detectados de {len(df)} velas.")
swing_points["HighLow"].value_counts()

1035 swing points detectados de 18630 velas.


HighLow
-1.0    518
 1.0    517
Name: count, dtype: int64

In [5]:
def render_swing_example(df: pd.DataFrame, swing_ts, swing_row, bars_of_context: int, out_path: Path) -> None:
    """Renderiza velas alrededor de un swing point con el marcador correspondiente."""
    pos = df.index.get_loc(swing_ts)
    lo = max(0, pos - bars_of_context)
    hi = min(len(df), pos + bars_of_context)
    window = df.iloc[lo:hi]

    marker_series = pd.Series(index=window.index, dtype=float)
    marker_series.loc[swing_ts] = swing_row["Level"]
    is_high = swing_row["HighLow"] == 1

    marker_plot = mpf.make_addplot(
        marker_series,
        type="scatter",
        marker="v" if is_high else "^",
        markersize=120,
        color="red" if is_high else "green",
    )

    label = "SWING HIGH" if is_high else "SWING LOW"
    mpf.plot(
        window,
        type="candle",
        style="yahoo",
        addplot=marker_plot,
        title=f"{SYMBOL} {TIMEFRAME} — {label} @ {swing_ts}",
        savefig=dict(fname=str(out_path), dpi=100),
    )
    plt.close("all")

In [6]:
import numpy as np

# Muestra N_EXAMPLES swing points repartidos a lo largo de todo el histórico (no solo los últimos),
# para ver variedad de condiciones de mercado.
sample_positions = np.linspace(0, len(swing_points) - 1, num=min(N_EXAMPLES, len(swing_points)), dtype=int)
sampled = swing_points.iloc[sample_positions]

for i, (ts, row) in enumerate(sampled.iterrows()):
    out_path = OUTPUT_DIR / f"{SYMBOL}_{TIMEFRAME}_{i:03d}.png"
    render_swing_example(df, ts, row, BARS_OF_CONTEXT, out_path)

print(f"{len(sampled)} imágenes guardadas en {OUTPUT_DIR}")

25 imágenes guardadas en C:\Users\isaac\Downloads\Projecto de trading IA\notebooks\00_rule_spec\output\swing_points


## Siguiente paso

Abre las imágenes en `notebooks/00_rule_spec/output/swing_points/` y revísalas: ¿los triángulos marcan swing highs/lows como los definirías tú? Si `SWING_LENGTH` está muy bajo vas a ver swings "de ruido" (demasiado sensibles); si está muy alto vas a perder swings menores que sí consideras válidos.

Cuando quede acordado, el valor de `SWING_LENGTH` (y si hace falta, por símbolo/timeframe) se escribe en `config/strategy_rules.yaml` bajo `swings.lookback_bars`, y el concepto pasa a **FORMALIZADO** en `docs/rule_decisions.md`.